# Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import col, trim
from pyspark.sql.window import Window

# Reading from bronze table

In [0]:
df = spark.table("workspace.bronze.crm_prd_info")

# Data transformations

## Renaming columns

In [0]:
RENAME_MAP = {
    "prd_id": "product_id",
    "cat_id": "category_id",
    "prd_key": "product_number",
    "prd_nm": "product_name",
    "prd_cost": "product_cost",
    "prd_line": "product_line",
    "prd_start_dt": "start_date",
    "prd_end_dt": "end_date"
}

for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

## Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

## Product key parsing

In [0]:
df = (
    df
    .withColumn(
        "category_id", 
        F.regexp_replace(F.substring(col("product_number"), 1, 5), "-", "_")
    )
    .withColumn(
        "product_number", 
        F.substring(col("product_number"), 7, F.length(col("product_number")))
    )
)

## Cost cleanup

In [0]:
df = (
    df
    .withColumn(
        "product_cost", 
        F.coalesce(col("product_cost"), F.lit(0))
    )
)

## Product line normalization

In [0]:
df = (
    df
    # Normalize product line
    .withColumn(
        "product_line",
        F.when(F.upper(col("product_line")) == "M", "Mountain")
         .when(F.upper(col("product_line")) == "R", "Road")
         .when(F.upper(col("product_line")) == "S", "Other Sales")
         .when(F.upper(col("product_line")) == "T", "Touring")
         .otherwise("n/a")
    )
)

## Date casting

In [0]:
df = (
    df
    .withColumn(
        "start_date", 
        col("start_date").cast(DateType())
    )
)

## Sanity check of final DataFrame

In [0]:
df.limit(10).display()

# Write into silver table

In [0]:
(
    df.write
        .mode("overwrite")
        .format("delta")
        .saveAsTable("workspace.silver.crm_products")
)